# Project Touchdown, stage 1: teach an AI to land a rocket

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArseniyCherednichenko/project-touchdown/blob/main/notebooks/stage1_land_a_rocket.ipynb)

In this notebook you train an AI to land a rocket, from zero, in your browser.
Nobody tells it how to fly. It only gets points, and it figures out the rest
by trying again and again.

Run the cells one by one, top to bottom (click a cell, then Shift + Enter).

Part of [Project Touchdown](https://github.com/ArseniyCherednichenko/project-touchdown)
by Arsenii Cherednichenko.

## 1. Install the tools
Takes about a minute. You only need to do this once.

In [ ]:
!pip install -q "gymnasium[box2d]" stable-baselines3 imageio imageio-ffmpeg

## 2. Meet the rocket

The simulator is **Lunar Lander**. It has gravity, a main engine pointing down
and two small side engines. The goal: land between the two flags.

Every frame, the simulator describes the rocket with **8 numbers**. This is
everything the AI will ever see. It never sees the picture.

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("LunarLander-v3", render_mode="rgb_array")
obs, _ = env.reset(seed=0)

names = ["x position", "y position (height)", "x speed", "y speed",
         "tilt", "turning speed", "left leg touching", "right leg touching"]
for name, value in zip(names, obs):
    print(f"{name:<22} {value:+.2f}")

And every frame, it can do **one of 4 things**:

| Action | What it does |
|---|---|
| 0 | nothing |
| 1 | fire the left side engine (pushes it right) |
| 2 | fire the main engine (pushes it up) |
| 3 | fire the right side engine (pushes it left) |

Here is what the rocket looks like at the start:

In [ ]:
from PIL import Image
Image.fromarray(env.render())

## 3. A pilot with no brain

Before training, let's see what happens if we just press random buttons.
`show()` turns the frames into a video.

In [ ]:
import base64, tempfile, imageio
from IPython.display import HTML

def show(frames, fps=50):
    path = tempfile.mktemp(suffix=".mp4")
    imageio.mimsave(path, frames, fps=fps, macro_block_size=8)
    data = base64.b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width="480" controls autoplay loop '
                f'src="data:video/mp4;base64,{data}"></video>')

def fly(pilot, seed=0):
    """Fly one attempt. pilot(obs) -> action. Returns frames and score."""
    obs, _ = env.reset(seed=seed)
    frames, score, done = [env.render()], 0.0, False
    while not done:
        obs, points, terminated, truncated, _ = env.step(pilot(obs))
        score += points
        frames.append(env.render())
        done = terminated or truncated
    return frames, score

random_pilot = lambda obs: env.action_space.sample()
frames, score = fly(random_pilot)
print(f"Score: {score:.0f}")
show(frames)

It crashes. Run the cell again a few times: different crash, similar score.

## 4. The points

The AI never gets told *how* to land. It only gets points, every frame:

| What happens | Points |
|---|---|
| Comes to rest safely | **+100** |
| Crashes or flies away | **-100** |
| Gets closer to the pad, slower, more level | a few points |
| Moves away, faster, more tilted | a few points off |
| A leg touches the ground | +10 per leg |
| Fires the main engine | -0.3 per frame |
| Fires a side engine | -0.03 per frame |

The score is all of these added up over one attempt. 200 or more counts as
landed well.

Why can the score go below -100 if a crash is only -100? Because all the
small minus points before the crash add up too.

In [ ]:
scores = [fly(random_pilot, seed=s)[1] for s in range(10)]
print("Random pilot, 10 attempts:", [round(s) for s in scores])

## 5. Give it a brain

The brain is a small **neural network**: 8 numbers go in, and out comes how
likely each of the 4 actions is. At first its guesses are random.

It learns with **PPO** (Proximal Policy Optimization), the same family of
methods used to train robots. The idea in one sentence: **try things, then
make the actions that led to more points a bit more likely, and the ones that
led to fewer points a bit less likely.** Repeat thousands of times.

Two settings worth knowing:
- `gamma=0.999`: how much it cares about points far in the future. Close to 1
  means "a landing in 10 seconds still matters now".
- `ent_coef=0.01`: a small push to keep trying new things instead of
  repeating the first thing that sort of worked.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

# 16 copies of the simulator run side by side, so it learns faster.
train_env = make_vec_env("LunarLander-v3", n_envs=16)

model = PPO("MlpPolicy", train_env, n_steps=1024, batch_size=64,
            gae_lambda=0.98, gamma=0.999, n_epochs=4, ent_coef=0.01, seed=0)
print(model.policy.mlp_extractor)

The printout shows two small networks, each with 2 layers of 64 neurons:
- **policy_net** is the pilot: it picks the action.
- **value_net** is a coach: it guesses how many points are still coming from
  here. Comparing that guess with what really happened tells PPO whether an
  action was better or worse than expected.

## 6. Train it

Now it practices. Every line below is an update: how many attempts so far,
the average score of the last 100, and how long the attempts lasted.

Every 100,000 steps you also get a short clip of how the AI flies at that
moment, played at 2x speed. Scroll down while it trains and watch it learn.

**Watch the "length" column.** The time limit is 1000 frames. If attempts get
close to 1000, the AI is hovering instead of landing: it found out that not
landing means not crashing. Keep watching, it usually grows out of it.

Time: 1,000,000 steps took 23 minutes on free Colab (measured, without the
clips) and 9 to 11 minutes on my laptop. The clips add a little on top.

In a hurry? `TRAIN_STEPS = 500_000` takes half the time. In my runs the AI
at 500,000 steps already landed most attempts, just less cleanly. You can
also skip ahead and load my finished AI in step 8.

In [ ]:
from IPython.display import display

TRAIN_STEPS = 1_000_000
CLIP_EVERY = 100_000  # steps between "how does it fly now?" clips

class Progress(BaseCallback):
    def __init__(self):
        super().__init__()
        self.scores, self.lengths = [], []
        self.next_clip = CLIP_EVERY
    def _on_step(self):
        for info in self.locals["infos"]:
            if "episode" in info:
                self.scores.append(info["episode"]["r"])
                self.lengths.append(info["episode"]["l"])
        if self.n_calls % 3000 == 0 and self.scores:
            print(f"attempts {len(self.scores):>5}   "
                  f"score {np.mean(self.scores[-100:]):>6.0f}   "
                  f"length {np.mean(self.lengths[-100:]):>5.0f}")
        if self.num_timesteps >= self.next_clip:
            self.next_clip += CLIP_EVERY
            pilot = lambda obs: self.model.predict(obs, deterministic=True)[0]
            frames, score = fly(pilot, seed=1)
            print(f"--- after {self.num_timesteps:,} steps: score {score:.0f} ---")
            display(show(frames[::2], fps=50))  # every 2nd frame = 2x speed
        return True

progress = Progress()
model.learn(total_timesteps=TRAIN_STEPS, callback=progress)

### How it learned

Every dot is one attempt. The line is the average of the last 100.

In [ ]:
import matplotlib.pyplot as plt

def smooth(x, n=100):
    return [np.mean(x[max(0, i - n):i + 1]) for i in range(len(x))]

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for ax, data, label in [(top, progress.scores, "Score"),
                        (bottom, progress.lengths, "Length (frames)")]:
    ax.scatter(range(len(data)), data, s=2, color="#2a78d6", alpha=0.15)
    ax.plot(smooth(data), color="#2a78d6", lw=2)
    ax.set_ylabel(label)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e8e7e3")
top.axhline(200, color="#52514e", ls="--", lw=1)
top.text(len(progress.scores), 205, "landed well", ha="right", va="bottom", color="#52514e")
bottom.axhline(1000, color="#52514e", ls="--", lw=1)
bottom.text(len(progress.lengths), 960, "time limit: hovering", ha="right", va="top", color="#52514e")
bottom.set_xlabel("Attempts")
plt.tight_layout()
plt.show()

## 7. Watch your AI land

In [ ]:
my_pilot = lambda obs: model.predict(obs, deterministic=True)[0]
frames, score = fly(my_pilot, seed=1)
print(f"Score: {score:.0f}")
show(frames)

In [ ]:
scores = [fly(my_pilot, seed=s)[1] for s in range(20)]
print(f"Landed well (200+): {sum(s >= 200 for s in scores)} of 20, "
      f"average {np.mean(scores):.0f}")

## 8. Compare with my AI

This downloads the AI I trained for Project Touchdown. It lands 19 out of 20
times.

In [ ]:
import urllib.request
url = ("https://raw.githubusercontent.com/ArseniyCherednichenko/"
       "project-touchdown/main/models/lander_ppo.zip")
urllib.request.urlretrieve(url, "lander_ppo.zip")
mine = PPO.load("lander_ppo.zip")
mine_pilot = lambda obs: mine.predict(obs, deterministic=True)[0]
frames, score = fly(mine_pilot, seed=1)
print(f"Score: {score:.0f}")
show(frames)

## What I learned

- The AI does exactly what the points reward, not what you meant. When
  crashing was expensive and landing was risky, hovering forever was the
  "smart" answer, until it learned that a real landing pays more.
- Nobody wrote a single rule about flying. Everything came from trial,
  error and points.
- Most of the work is not the AI. It is deciding what to reward.

## Try it yourself

- Change `gamma` to `0.9` and train again. Does it still land? Why might
  caring less about the future make landing harder?
- Change `ent_coef` to `0`. Does it get stuck more?
- **Can you beat my score?** My AI averages about 240. If you beat it,
  [tell me how](https://github.com/ArseniyCherednichenko/project-touchdown/issues).

Next, in stage 2: what happens when you change the points.